
###Dependência externa — biblioteca holidays

Necessária para gerar_df_feriados_brasil() (Frente D). Instalada via %pip no próprio notebook — padrão Databricks para dependências de notebook, em vez de depender de configuração manual do cluster (que é justamente o tipo de passo manual que o plano de correção quer eliminar). %restart_python garante que a lib fique disponível na sessão atual imediatamente após a instalação.

Bug corrigido: a versão anterior usava %pip install + %restart_python incondicionalmente. Como 00_setup_ibge também roda %run "../utils/00_utils" internamente, um notebook que chama os dois em sequência (ex.: gold_physical_lojas → 00_utils → depois → 00_setup_ibge → 00_utils de novo) disparava o restart duas vezes — a segunda vez já depois de imports/variáveis já terem sido definidos em células anteriores do notebook pai, apagando tudo (NameError: name 'expr' is not defined e afins).

Agora só instala e reinicia se a lib ainda não estiver disponível — usando dbutils.library.restartPython() dentro de um try/except (diferente do %restart_python, que roda sempre, sem condição, toda vez que a célula é executada — mesmo que a lib já exista).

In [0]:
try:
    import holidays
    print("[OK] Biblioteca 'holidays' já disponível — sem reinstalar/reiniciar.")
except ImportError:
    print("[INFO] Biblioteca 'holidays' ausente — instalando...")
    %pip install holidays --quiet
    dbutils.library.restartPython()

In [0]:
# ══════════════════════════════════════
# 00_UTILS — Utilitários do Pipeline
# Squad 3 — Arquitetura Medalhão
# Batch Lojas Físicas
# Luiz Henrique Portácio
# ══════════════════════════════════════

# Utilitários — Squad 3 (Batch Lojas Físicas)

Funções reutilizáveis usadas pelos notebooks de Bronze, Silver, Gold
e Analysis. Importado via `%run` no início de cada notebook do
pipeline. Carrega também `governanca/00_data_quality_rules`, pois
algumas funções abaixo dependem de `DQ_RULES_VERSION`.

**Sobre autenticação no ADLS Gen2 (Serverless):**
No Databricks Serverless, `spark.conf.set()` para propriedades
`fs.azure.account.*` não é permitido (`CONFIG_NOT_AVAILABLE`).
A solução é usar `build_adls_options()` para montar um dicionário
de credenciais OAuth e passá-lo via `.options(**adls_options)` em
cada operação `spark.read` / `spark.write`. As credenciais vêm das
variáveis definidas no `00_config` (lidas do `.env`).

In [0]:
%run "../config/00_config"

In [0]:
%run "../governanca/00_data_quality_rules"

In [0]:
import uuid
from datetime import datetime, timezone

from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, lit, current_timestamp, count, when, udf
)
from pyspark.sql.types import StringType
from pyspark.sql.utils import AnalysisException

####Feriados gerados automaticamente (sem passo manual)

Diferente do enriquecimento IBGE (que depende de uma API externa e hoje é um notebook manual, 00_setup_ibge), os feriados nacionais brasileiros podem ser calculados deterministicamente, sem chamada de rede, usando a biblioteca holidays. Isso inclui feriados móveis (Sexta-feira Santa, Corpus Christi, Carnaval) calculados a partir da data da Páscoa.

Isso elimina de vez a dependência de uma tabela Silver feriados gerada manualmente/externamente (que hoje não tem notebook nenhum responsável por escrevê-la — se não existir, o pipeline de Gold simplesmente ignora a flag em silêncio via except).

Requer: pip install holidays no cluster/ambiente Databricks.

In [0]:
import holidays


def gerar_df_feriados_brasil(spark, anos, uf=None):
    """
    Gera um DataFrame Spark com todos os feriados nacionais
    brasileiros (e estaduais, se `uf` for informado) para os anos
    dados, calculados deterministicamente via biblioteca `holidays` —
    sem dependência de API externa nem de passo manual.

    Parâmetros:
        spark: SparkSession ativa.
        anos: lista de anos (ex.: [2024, 2025, 2026]).
        uf: sigla do estado (ex.: "SP") para incluir feriados
            estaduais/municipais além dos nacionais. Se None,
            retorna só feriados nacionais.

    Retorna:
        DataFrame com colunas: data_feriado (date), nome_feriado (string).
    """
    calendario = holidays.Brazil(years=anos, subdiv=uf)

    linhas = [
        (data, nome)
        for data, nome in sorted(calendario.items())
    ]

    df_feriados = spark.createDataFrame(
        linhas, schema="data_feriado date, nome_feriado string"
    )

    return df_feriados

## Autenticação ADLS Gen2 — OAuth via Service Principal

No Serverless, as credenciais OAuth são passadas como opções
diretamente em cada operação Spark (`.options(**adls_options)`),
não como configuração global via `spark.conf.set()`.

In [0]:
def build_adls_options(storage_account_name, client_id, tenant_id, client_secret):
    """
    Monta o dicionário de opções OAuth para autenticação no ADLS Gen2
    via Service Principal. Compatível com Databricks Serverless (onde
    spark.conf.set para fs.azure.account.* não é permitido).

    Uso: spark.read.options(**adls_options).format("delta").load(path)
    """
    return {
        f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net":
            "OAuth",
        f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net":
            "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
        f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net":
            client_id,
        f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net":
            client_secret,
        f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net":
            f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
    }


def get_adls_options():
    """
    Retorna as opções OAuth prontas para uso, usando as credenciais
    carregadas pelo 00_config a partir do .env.
    """
    return build_adls_options(
        ADLS_STORAGE_ACCOUNT_NAME,
        ADLS_CLIENT_ID,
        ADLS_TENANT_ID,
        ADLS_CLIENT_SECRET,
    )

## Leitura e escrita no ADLS

In [0]:
def read_delta(path, adls_options):
    """Lê uma tabela Delta do ADLS, autenticando via adls_options."""
    return (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(path)
    )


def write_delta(df, path, adls_options, mode="append", merge_schema=True,
                partition_by=None, overwrite_schema=False):
    """
    Grava um DataFrame em Delta no ADLS, autenticando via adls_options.

    Parâmetros:
        mode: "append" (padrão para Bronze) ou "overwrite" (Gold/Silver)
        merge_schema: se True, permite evolução de schema (novas colunas)
        partition_by: lista de colunas para particionar (opcional)
        overwrite_schema: se True, substitui o schema completamente —
                          necessário quando há conflito de TIPO em colunas
                          existentes (ex: LongType vs StringType)
    """
    writer = (
        df.write
        .format("delta")
        .options(**adls_options)
        .option("mergeSchema", str(merge_schema).lower())
        .option("overwriteSchema", str(overwrite_schema).lower())
        .mode(mode)
    )
    if partition_by:
        writer = writer.partitionBy(*partition_by)
    writer.save(path)


def read_source_csv(spark, source_path, adls_options, csv_options=None):
    """
    Lê um CSV da Raw, autenticando via adls_options.
    Assinatura idêntica ao padrão da squad (colega Ygor Moraes).

    Parâmetros:
        spark: SparkSession ativa
        source_path: caminho abfss:// do arquivo CSV
        adls_options: dicionário OAuth retornado por get_adls_options()
        csv_options: dicionário com opções do CSV (ex: header, inferSchema)
    """
    csv_options = csv_options or {"header": "true", "inferSchema": "false"}
    return (
        spark.read
        .format("csv")
        .options(**csv_options)
        .options(**adls_options)
        .load(source_path)
    )

## Verificação de pré-condição

In [0]:
def verificar_destino_limpo(path, adls_options, permitir_existente=False):
    """
    Verifica se um caminho Delta já existe e contém dados.
    Interrompe a execução com erro claro se existir e
    permitir_existente=False, evitando escrita sobre resíduos.
    """
    try:
        df_existente = read_delta(path, adls_options)
        contagem = df_existente.count()
    except Exception:
        print(f"[OK] Caminho '{path}' não existe ainda. Pronto para escrita limpa.")
        return

    if contagem > 0 and not permitir_existente:
        raise RuntimeError(
            f"ERRO DE PRÉ-CONDIÇÃO: o caminho '{path}' já contém "
            f"{contagem} linha(s). Confirme que a limpeza do ambiente foi "
            f"concluída, ou chame com permitir_existente=True para append."
        )

    print(f"[OK] Caminho '{path}' — {contagem} linha(s) existentes.")

## Camada Bronze — cast para string e metadados de auditoria

In [0]:
def cast_all_columns_to_string(df):
    """
    Converte todas as colunas do DataFrame para STRING.
    Uso exclusivo da camada Bronze.
    """
    return df.select([col(c).cast("string").alias(c) for c in df.columns])


def adicionar_metadados_bronze(df, batch_id=None, source_file=None):
    """
    Adiciona colunas de auditoria/rastreabilidade à camada Bronze.
    Essas colunas existem exclusivamente no Delta da Bronze.
    """
    batch_id = batch_id or str(uuid.uuid4())
    df = df.withColumn("bronze_batch_id", lit(batch_id))
    df = df.withColumn("bronze_ingested_at", current_timestamp())
    if source_file:
        df = df.withColumn("bronze_source_file", lit(source_file))
    return df


def get_latest_batch_id(path, adls_options):
    """
    Retorna o bronze_batch_id mais recente em um caminho Delta da Bronze.
    """
    df = read_delta(path, adls_options)
    ultimo = (
        df.select("bronze_batch_id", "bronze_ingested_at")
        .orderBy(col("bronze_ingested_at").desc())
        .limit(1)
        .collect()
    )
    if not ultimo:
        raise RuntimeError(f"Nenhum batch encontrado em '{path}'.")
    return ultimo[0]["bronze_batch_id"]

## Camada Silver — metadados, quarentena e métricas de qualidade

In [0]:
def adicionar_metadados_silver(df):
    """Adiciona colunas de auditoria à camada Silver."""
    return (
        df
        .withColumn("silver_processed_at", current_timestamp())
        .withColumn("dq_rules_version", lit(DQ_RULES_VERSION))
    )


def separar_quarentena_pk(df, coluna_pk, quarentena_path, adls_options):
    """
    Identifica registros com PK nula ou duplicada e os move para
    quarentena (Delta, exclusiva da Silver). Retorna apenas os válidos.

    Performance (bug corrigido): a versão anterior fazia
    groupBy(pk).count().collect() para achar as PKs duplicadas e depois
    um .isin(lista_grande) para filtrar -- com PK de altissima
    cardinalidade (ex.: id_item_venda, quase 1 valor por linha) isso gera
    uma expressao de filtro enorme que pode travar o driver/kernel por
    completo, nao so deixar lento. A versao abaixo aplica row_number()
    diretamente sobre TODO o dataset valido (particao por PK, ordenado
    por bronze_ingested_at desc) em uma unica passada -- sem collect()
    para o driver e sem isin() com lista grande.
    """
    from pyspark.sql.functions import row_number
    from pyspark.sql.window import Window

    total = df.count()
    df_pk_nula = df.filter(col(coluna_pk).isNull())
    nulos = df_pk_nula.count()

    if total > 0 and nulos == total:
        raise RuntimeError(
            f"ERRO FATAL: 100% dos registros têm '{coluna_pk}' nula. "
            f"Problema estrutural na origem. Pipeline interrompido."
        )

    df_validos_base = df.filter(col(coluna_pk).isNotNull())

    # row_number = 1 cobre tanto os registros UNICOS (nao duplicados)
    # quanto o "vencedor" mais recente de cada PK duplicada -- nao
    # precisa identificar duplicatas antecipadamente para isso funcionar.
    janela_recente = Window.partitionBy(coluna_pk).orderBy(
        col("bronze_ingested_at").desc()
    )
    df_ranked = df_validos_base.withColumn(
        "_rn", row_number().over(janela_recente)
    )

    df_validos   = df_ranked.filter(col("_rn") == 1).drop("_rn")
    df_excedente = df_ranked.filter(col("_rn") > 1).drop("_rn")

    df_quarentena = df_pk_nula.unionByName(df_excedente, allowMissingColumns=True)

    qtd_quarentena = df_quarentena.count()
    if qtd_quarentena > 0:
        df_q = (
            df_quarentena
            .withColumn(
                "quarentena_motivo",
                when(col(coluna_pk).isNull(), lit("pk_nula"))
                .otherwise(lit("pk_duplicada"))
            )
            .withColumn("quarentena_registrado_em", current_timestamp())
        )
        write_delta(df_q, quarentena_path, adls_options, mode="append",
                    merge_schema=True)
        print(f"[QUARENTENA] {qtd_quarentena} registro(s) -> '{quarentena_path}'.")

    return df_validos


def registrar_metrica_dq(tabela, regra, qtd_registros_afetados,
                          qtd_registros_total, adls_options,
                          metrics_path=None, execucao_id=None):
    """
    Registra métricas de qualidade centralizadas (fonte dos gráficos
    nos notebooks Analysis). Exclusivo do Delta da Silver.

    Frente D (bug corrigido): antes gravava sempre em modo "append",
    acumulando indefinidamente TODAS as execucoes historicas do
    pipeline nas mesmas linhas -- o consumidor (Analysis/Looker) nao
    tinha como saber qual e o snapshot da ULTIMA execucao sem logica
    extra. Agora cada linha carrega um `execucao_id` (mesmo timestamp
    de execucao para todas as regras rodadas na mesma chamada do
    pipeline), permitindo filtrar so a execucao mais recente com
    `MAX(execucao_id)` -- sem precisar mudar o modo de escrita para
    "overwrite" (que perderia o historico, util para auditoria).
    """
    metrics_path = metrics_path or SILVER_DQ_METRICS_PATH
    execucao_id = execucao_id or datetime.now(timezone.utc)
    linha = spark.createDataFrame(
        [(
            tabela,
            regra,
            qtd_registros_afetados,
            qtd_registros_total,
            DQ_RULES_VERSION,
            datetime.now(timezone.utc),
            execucao_id,
        )],
        schema=(
            "tabela string, regra string, qtd_afetados long, "
            "qtd_total long, dq_rules_version string, registrado_em timestamp, "
            "execucao_id timestamp"
        ),
    )
    write_delta(linha, metrics_path, adls_options, mode="append", merge_schema=True)


## Camada Gold — escrita tipada no SQL Server

Uso **exclusivo** da camada Gold. Bronze e Silver nunca chamam
esta função.

In [0]:
def write_sql_table_typed(df, table_name, mode="overwrite"):
    """
    Grava um DataFrame com tipos reais no SQL Server Azure,
    no schema definido em TARGET_SCHEMA. Uso exclusivo da Gold.

    Performance (bug corrigido 2x): a primeira tentativa usou
    bulkCopyBatchSize/bulkCopyTableLock/bulkCopyTimeout -- opcoes do
    modo bulk-copy do conector, mas o Databricks Serverless BLOQUEIA
    essas opcoes especificas (DATA_SOURCE_OPTIONS_VALIDATION_FAILED).
    A lista de opcoes permitidas em Serverless inclui apenas "batchsize"
    e "numPartitions" para tuning de performance -- usamos essas duas,
    que ainda reduzem bastante o numero de round-trips ao banco em
    relacao ao batch padrao (pequeno), sem exigir o modo bulk-copy
    completo que o Serverless nao permite.
    """
    total_linhas = df.count()  # 1x so, reaproveitado no print final

    (
        df.write
        .format("sqlserver")
        .option("host", SQL_HOST)
        .option("port", SQL_PORT)
        .option("database", SQL_DATABASE)
        .option("dbtable", table_name)
        .option("user", SQL_USERNAME)
        .option("password", SQL_PASSWORD)
        .option("batchsize", "10000")
        .option("numPartitions", "8")
        .mode(mode)
        .save()
    )
    print(f"[OK] '{table_name}' gravada no SQL Server ({mode}), {total_linhas:,} linha(s).")


print("00_utils carregado com sucesso.")